In [3]:
import os
import librosa
import numpy as np
import pandas as pd

# ---------------------- 配置参数（全部是绝对路径！） ----------------------
# 拆分后的音轨根目录
SEPARATED_DIR = "/Users/xiahanfei/Desktop/MGS3001/AI_music/songs download/songs/separated"
# 你已有的AIGC率表格路径
INPUT_CSV = "/Users/xiahanfei/Desktop/MGS3001/AI_music/AIGC rate/aigc_rates.csv"
# 输出结果路径
OUTPUT_CSV = "/Users/xiahanfei/Desktop/MGS3001/AI_music/permeability_results.csv"

# 各个环节的权重（和我们的理论框架完全一致）
WEIGHTS = {
    'melody': 0.20,      # 旋律
    'lyric': 0.20,       # 歌词
    'harmony': 0.15,     # 和声
    'structure': 0.10,   # 结构
    'arrangement': 0.15, # 编曲
    'vocal': 0.10,       # 人声
    'mix': 0.05,         # 混音
    'master': 0.05       # 母带
}

# ---------------------- 工具函数 ----------------------
def calibrate_depth(score, has_modification=False, is_inspiration=False):
    """
    将AIGC检测分数校准为真实的AI参与深度d_i
    """
    if is_inspiration:
        return 0.25
    if score <= 0:
        return 0.0
    if score > 0.9 and not has_modification:
        return 1.0
    elif 0.6 <= score <= 0.9 and has_modification:
        return 0.75
    elif 0.3 <= score <= 0.6:
        return 0.5
    else:
        return min(score, 1.0)  # 默认直接用分数作为深度

def extract_features(accompaniment_path):
    """
    从伴奏音轨中提取旋律、和声、结构特征
    """
    try:
        # 加载伴奏音轨
        y, sr = librosa.load(accompaniment_path, sr=22050)
        
        # 1. 提取旋律特征
        melody = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))[0]
        
        # 2. 提取和声/和弦特征
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        
        # 3. 提取歌曲结构特征
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        
        return {
            'melody_feature': melody,
            'harmony_feature': chroma,
            'structure_feature': {
                'tempo': tempo,
                'spectral_contrast': spectral_contrast
            }
        }
    except Exception as e:
        print(f"特征提取失败: {e}")
        return None

# ---------------------- 主处理逻辑 ----------------------
def main():
    # 1. 读取你已有的AIGC率数据
    print("加载已有的AIGC率数据...")
    if not os.path.exists(INPUT_CSV):
        print(f"错误：找不到 {INPUT_CSV} 文件！")
        return
        
    df = pd.read_csv(INPUT_CSV)
    print(f"加载了 {len(df)} 首歌的基础数据")
    
    # 2. 遍历所有拆分好的歌曲
    results = []
    processed = 0
    
    for _, row in df.iterrows():
        artist = row['artist_name']
        song = row['song_name']
        full_song_name = f"{artist} - {song}"
        print(f"\n正在处理: {full_song_name}")
        
        # 找到对应的wav文件！现在用完整的文件名前缀了！
        vocal_path = os.path.join(SEPARATED_DIR, f"{full_song_name}_vocals.wav")
        accomp_path = os.path.join(SEPARATED_DIR, f"{full_song_name}_accompaniment.wav")
        
        if not os.path.exists(vocal_path):
            print(f"未找到人声音轨: {vocal_path}")
            continue
        if not os.path.exists(accomp_path):
            print(f"未找到伴奏音轨: {accomp_path}")
            continue
        
        print(f"找到文件，开始提取特征...")
        # 3. 提取特征
        features = extract_features(accomp_path)
        if not features:
            continue
        
        # 4. 获取各个环节的AIGC率
        total_audio_rate = row['Audio_AIGC_Rate']
        lyric_rate = row['Lyrics_AIGC_Rate']
        
        # 拆分音频的分环节率
        vocal_rate = total_audio_rate * 0.3  # 人声占30%
        arrangement_rate = total_audio_rate * 0.7  # 编曲占70%
        melody_rate = arrangement_rate * 0.4  # 旋律占编曲的40%
        harmony_rate = arrangement_rate * 0.3  # 和声占编曲的30%
        structure_rate = arrangement_rate * 0.3  # 结构占编曲的30%
        mix_rate = total_audio_rate * 0.1  # 混音
        master_rate = total_audio_rate * 0.1  # 母带
        
        # 5. 校准每个环节的参与深度
        d_melody = calibrate_depth(melody_rate)
        d_lyric = calibrate_depth(lyric_rate)
        d_harmony = calibrate_depth(harmony_rate)
        d_structure = calibrate_depth(structure_rate)
        d_arrangement = calibrate_depth(arrangement_rate)
        d_vocal = calibrate_depth(vocal_rate)
        d_mix = calibrate_depth(mix_rate)
        d_master = calibrate_depth(master_rate)
        
        # 6. 计算最终的AI渗透度
        permeability = (
            WEIGHTS['melody'] * d_melody +
            WEIGHTS['lyric'] * d_lyric +
            WEIGHTS['harmony'] * d_harmony +
            WEIGHTS['structure'] * d_structure +
            WEIGHTS['arrangement'] * d_arrangement +
            WEIGHTS['vocal'] * d_vocal +
            WEIGHTS['mix'] * d_mix +
            WEIGHTS['master'] * d_master
        )
        
        # 7. 保存结果，保留所有原始信息
        results.append({
            'music_genre': row['music_genre'],
            'artist_name': artist,
            'song_name': song,
            'original_audio_aigc': total_audio_rate,
            'original_lyric_aigc': lyric_rate,
            'melody_aigc': melody_rate,
            'harmony_aigc': harmony_rate,
            'structure_aigc': structure_rate,
            'arrangement_aigc': arrangement_rate,
            'vocal_aigc': vocal_rate,
            'mix_aigc': mix_rate,
            'master_aigc': master_rate,
            'ai_permeability': permeability
        })
        
        processed += 1
        print(f"处理完成，AI渗透度: {permeability:.3f}")
    
    # 8. 保存结果到CSV
    if len(results) > 0:
        result_df = pd.DataFrame(results)
        result_df.to_csv(OUTPUT_CSV, index=False)
        
        print(f"\n==================== 全部完成 ====================")
        print(f"共处理了 {processed} 首歌")
        print(f"结果已保存到: {OUTPUT_CSV}")
        print(f"平均AI渗透度: {result_df['ai_permeability'].mean():.3f}")
    else:
        print("没有处理任何歌曲，请检查文件路径是否正确！")

if __name__ == "__main__":
    main()

加载已有的AIGC率数据...
加载了 575 首歌的基础数据

正在处理: 郑润泽 - Intro
找到文件，开始提取特征...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


处理完成，AI渗透度: 0.039

正在处理: 郑润泽 - Outro
找到文件，开始提取特征...
处理完成，AI渗透度: 0.055

正在处理: 郑润泽 - 任性
找到文件，开始提取特征...
处理完成，AI渗透度: 0.060

正在处理: 郑润泽 - 是你
找到文件，开始提取特征...
处理完成，AI渗透度: 0.076

正在处理: 郑润泽 - 只在今夜
找到文件，开始提取特征...
处理完成，AI渗透度: 0.061

正在处理: 郑润泽 - 像晴天像雨天
找到文件，开始提取特征...
处理完成，AI渗透度: 0.062

正在处理: 郑润泽 - 晚点
找到文件，开始提取特征...
处理完成，AI渗透度: 0.060

正在处理: 郑润泽 - 倔强
找到文件，开始提取特征...
处理完成，AI渗透度: 0.060

正在处理: 郑润泽 - 隐形人
找到文件，开始提取特征...
处理完成，AI渗透度: 0.061

正在处理: 郑润泽 - 一切的一切
找到文件，开始提取特征...
处理完成，AI渗透度: 0.061

正在处理: 郑润泽 - 看着我
找到文件，开始提取特征...
处理完成，AI渗透度: 0.061

正在处理: 郑润泽 - Crush
找到文件，开始提取特征...
处理完成，AI渗透度: 0.061

正在处理: 郑润泽 - My Dear
找到文件，开始提取特征...
处理完成，AI渗透度: 0.062

正在处理: 郑润泽 - 想悄悄住进你的灵魂
找到文件，开始提取特征...
处理完成，AI渗透度: 0.091

正在处理: 郑润泽 - 雨滴中有你
找到文件，开始提取特征...
处理完成，AI渗透度: 0.061

正在处理: 郑润泽 - 追光自会成为光
找到文件，开始提取特征...
处理完成，AI渗透度: 0.063

正在处理: 郑润泽 - 纯白 (INTRO)
找到文件，开始提取特征...
处理完成，AI渗透度: 0.039

正在处理: 郑润泽 - 纯白
找到文件，开始提取特征...
处理完成，AI渗透度: 0.060

正在处理: 郑润泽 - 万物皆可爱 (INTRO)
找到文件，开始提取特征...
处理完成，AI渗透度: 0.051

正在处理: 郑润泽 - 万物皆可爱
找到文件，开始提取特征...
处理完成，AI渗透度

In [2]:
import pandas as pd

# 输入输出路径（和你之前的结果路径对应）
INPUT_CSV = "/Users/xiahanfei/Desktop/MGS3001/AI_music/permeability_results.csv"
OUTPUT_CSV = "/Users/xiahanfei/Desktop/MGS3001/AI_music/permeability_results_cleaned.csv"

# 读取原始结果数据
df = pd.read_csv(INPUT_CSV)

# 所有需要保留4位小数的数值列
numeric_columns = [
    'original_audio_aigc',
    'original_lyric_aigc',
    'melody_aigc',
    'harmony_aigc',
    'structure_aigc',
    'arrangement_aigc',
    'vocal_aigc',
    'mix_aigc',
    'master_aigc',
    'ai_permeability'
]

# 批量处理，所有数值保留4位小数
for col in numeric_columns:
    df[col] = df[col].round(3)

# 保存清洗后的结果
df.to_csv(OUTPUT_CSV, index=False)

print("="*30)
print("数据清洗完成！")
print(f"共处理了 {len(df)} 首歌曲的数据")
print(f"清洗后的文件已保存到: {OUTPUT_CSV}")
print("所有AIGC率和AI渗透度都已保留3位小数，原始文件未被修改。")

数据清洗完成！
共处理了 575 首歌曲的数据
清洗后的文件已保存到: /Users/xiahanfei/Desktop/MGS3001/AI_music/permeability_results_cleaned.csv
所有AIGC率和AI渗透度都已保留3位小数，原始文件未被修改。


In [6]:
import pandas as pd
import numpy as np

# 文件路径
INPUT_FILE = "/Users/xiahanfei/Desktop/MGS3001/AI_music/FINAL_ANALYSIS_DATA_FULL.csv"
OUTPUT_FILE = "/Users/xiahanfei/Desktop/MGS3001/AI_music/FINAL_ANALYSIS_DATA_CLEANED.csv"

print("="*60)
print("开始数据清洗...")
print(f"原始文件: {INPUT_FILE}")

# 1. 读取原始数据
df = pd.read_csv(INPUT_FILE)
original_count = len(df)
print(f"\n原始数据量: {original_count} 首歌曲")

# 2. 第一步：去重（按歌手+歌名，避免同一首歌重复）
df['_temp_key'] = df['artist_name'].astype(str).str.strip() + "_" + df['song_name'].astype(str).str.strip()
duplicate_count = df.duplicated(subset=['_temp_key']).sum()
if duplicate_count > 0:
    df = df.drop_duplicates(subset=['_temp_key'], keep='first')
    print(f"✅ 去除重复歌曲: {duplicate_count} 首")
else:
    print("✅ 没有发现重复歌曲")

# 3. 第二步：处理缺失值
# 核心数值列，这些列不能有空值
core_numeric_cols = [
    'original_audio_aigc', 'original_lyric_aigc',
    'melody_aigc', 'harmony_aigc', 'structure_aigc',
    'arrangement_aigc', 'vocal_aigc', 'mix_aigc', 'master_aigc',
    'ai_permeability'
]

# 统计缺失的行
missing_mask = df[core_numeric_cols].isna().any(axis=1)
missing_count = missing_mask.sum()
if missing_count > 0:
    df = df[~missing_mask]
    print(f"✅ 去除缺失核心数据的行: {missing_count} 首")
else:
    print("✅ 没有发现缺失核心数据的行")

# 4. 第三步：处理异常值（AIGC率和渗透度必须在0~1之间）
# 检查所有数值列有没有超出0~1的异常值
outlier_mask = pd.Series([False]*len(df), index=df.index)
for col in core_numeric_cols:
    # 找出小于0或者大于1的异常值
    col_outlier = (df[col] < 0) | (df[col] > 1)
    outlier_mask = outlier_mask | col_outlier

outlier_count = outlier_mask.sum()
if outlier_count > 0:
    df = df[~outlier_mask]
    print(f"✅ 去除数值异常的行: {outlier_count} 首（AIGC率/渗透度超出0~1范围）")
else:
    print("✅ 没有发现数值异常的行")

# 5. 删除临时列
df = df.drop(columns=['_temp_key'])

# 6. 保存清洗后的数据
df.to_csv(OUTPUT_FILE, index=False)

# 7. 最终统计
final_count = len(df)
removed_total = original_count - final_count

print("\n" + "="*60)
print("✅ 数据清洗完成！")
print(f"  原始数据: {original_count} 首")
print(f"  清洗后: {final_count} 首")
print(f"  共去除无效数据: {removed_total} 首（占比 {removed_total/original_count*100:.2f}%）")
print(f"\n  清洗后的文件已保存到: {OUTPUT_FILE}")
print("  这个文件已经没有脏数据了，可以直接用于你的学术分析~")

开始数据清洗...
原始文件: /Users/xiahanfei/Desktop/MGS3001/AI_music/FINAL_ANALYSIS_DATA_FULL.csv

原始数据量: 587 首歌曲
✅ 去除重复歌曲: 12 首
✅ 没有发现缺失核心数据的行
✅ 没有发现数值异常的行

✅ 数据清洗完成！
  原始数据: 587 首
  清洗后: 575 首
  共去除无效数据: 12 首（占比 2.04%）

  清洗后的文件已保存到: /Users/xiahanfei/Desktop/MGS3001/AI_music/FINAL_ANALYSIS_DATA_CLEANED.csv
  这个文件已经没有脏数据了，可以直接用于你的学术分析~
